<a href="https://colab.research.google.com/github/kasiranaweera/CDAZZDEV-MLE-KASI/blob/main/task3_agentic/CDAZZDEV_Task3_Agentic_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: Multi-Agent Financial Research System with Memory & Observability

Step 1 — Install Dependencies

In [ ]:
# STEP 1 — Install Dependencies
!pip install -q langchain langchain-groq langgraph langchain-community
!pip install -q yfinance duckduckgo-search pydantic
!pip install -q pandas numpy scipy
!pip install -q wandb

print("✅ All packages installed.")


Step 2 — Imports & Configuration

In [ ]:
# STEP 2 — Imports & Configuration
import os
import json
import time
import uuid
import operator
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Optional, Annotated, Sequence, TypedDict

import pandas as pd
import numpy as np
import yfinance as yf
from pydantic import BaseModel, Field, ValidationError

warnings.filterwarnings("ignore")

# LangChain / LangGraph
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import (
    BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage,
)
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent

# ---- API KEY ----
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("Empty secret")
except Exception:
    GROQ_API_KEY = input("🔑 Enter your Groq API key (free at console.groq.com): ").strip()

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ---- Constants ----
LLM_MODEL    = "llama-3.3-70b-versatile"   # Groq free tier — best reasoning
TRACE_FILE   = "agent_trace.jsonl"          # Required for Task 3C
CACHE_DIR    = Path("research_cache")
CACHE_DIR.mkdir(exist_ok=True)

DEFAULT_TICKER = "NVDA"   # ← Change this to any ticker you want

print("✅ Environment configured.")
print(f"   Model  : {LLM_MODEL}")
print(f"   Ticker : {DEFAULT_TICKER}")

Step 3 — Pydantic Schemas

In [ ]:
# STEP 3 — Pydantic Schemas
class HeadlineSentiment(BaseModel):
    headline: str
    sentiment: str          # positive | negative | neutral
    confidence: float = Field(ge=0.0, le=1.0)
    brief_reason: str

class SentimentResult(BaseModel):
    analyses: list[HeadlineSentiment]
    aggregate_score: float = Field(ge=-1.0, le=1.0)
    overall_sentiment: str

class RiskItem(BaseModel):
    risk: str
    supporting_evidence: str
    severity: str           # high | medium | low

class HedgeStrategy(BaseModel):
    strategy: str
    rationale: str
    instruments: list[str]

class DataBrief(BaseModel):
    """Structured handoff from Agent A → Agent B (Task 3B)."""
    ticker: str
    price_snapshot: dict
    volatility_metrics: dict
    sentiment_result: dict
    momentum_signal: str
    key_observations: list[str] = Field(default_factory=list)
    generated_at: str

class FinalReport(BaseModel):
    """Validated output of the full pipeline."""
    ticker: str
    financial_health_summary: str
    top_three_risks: list[RiskItem]
    hedge_strategy: HedgeStrategy
    sources_consulted: list[str]
    generated_at: str

print("✅ Pydantic schemas defined.")

Step 4 — Observability: Tracer + Session Memory

In [ ]:
# STEP 4 — Observability: Tracer + Session Memory
# AI-ASSISTED: Claude (claude-sonnet-4-6),
# Prompt: 'Design a JSONL observability tracer for LangChain agent tool calls',
# Date: 2026-05-11

_session_memory: dict[str, Any] = {}   # in-process short-term memory

class AgentTracer:
    """
    Logs every tool call to agent_trace.jsonl as required by Task 3C.
    Fields: session_id, timestamp, tool_name, inputs, output_truncated (≤200 chars), duration_seconds.
    """
    def __init__(self, path: str = TRACE_FILE):
        self.path = path
        self.session_id = str(uuid.uuid4())[:8]
        self._records: list[dict] = []

    def log(
        self,
        tool_name: str,
        inputs: dict,
        output: Any,
        duration: float,
    ) -> dict:
        record = {
            "session_id"       : self.session_id,
            "timestamp"        : datetime.utcnow().isoformat() + "Z",
            "tool_name"        : tool_name,
            "inputs"           : inputs,
            "output_truncated" : str(output)[:200],
            "duration_seconds" : round(duration, 4),
        }
        with open(self.path, "a") as fh:
            fh.write(json.dumps(record) + "\n")
        self._records.append(record)
        print(f"  📋 [{tool_name}] {duration:.2f}s | {str(output)[:60]}...")
        return record

    @property
    def all_records(self) -> list[dict]:
        return self._records

tracer = AgentTracer()
print(f"✅ Tracer active → {TRACE_FILE}  (session: {tracer.session_id})")

Step 5 — Technical Indicator Helpers (no TA-Lib)

In [ ]:
# STEP 5 — Technical Indicator Helpers (no TA-Lib)
def _sma(s: pd.Series, w: int) -> pd.Series:
    return s.rolling(w).mean()

def _rsi(s: pd.Series, p: int = 14) -> pd.Series:
    d   = s.diff()
    g   = d.clip(lower=0)
    l   = -d.clip(upper=0)
    ag  = g.ewm(com=p - 1, min_periods=p).mean()
    al  = l.ewm(com=p - 1, min_periods=p).mean()
    return 100 - 100 / (1 + ag / al.replace(0, np.nan))

def _macd(s: pd.Series, fast=12, slow=26, sig=9):
    ef = s.ewm(span=fast, adjust=False).mean()
    es = s.ewm(span=slow, adjust=False).mean()
    m  = ef - es
    sg = m.ewm(span=sig, adjust=False).mean()
    return m, sg, m - sg

def _bollinger(s: pd.Series, w=20, sd=2):
    mu = s.rolling(w).mean()
    st = s.rolling(w).std()
    return mu + sd * st, mu, mu - sd * st

def _f(v) -> Optional[float]:
    """Safe float — returns None for NaN/Inf."""
    try:
        fv = float(v)
        return None if (np.isnan(fv) or np.isinf(fv)) else round(fv, 4)
    except Exception:
        return None

print("✅ Indicator helpers ready.")

Step 6 — Five Tool Definitions

In [ ]:
# STEP 6 — Five Tool Definitions
# AI-ASSISTED: Claude (claude-sonnet-4-6),
# Prompt: 'Implement LangChain @tool wrappers for get_price_data, get_news,
#          calculate_volatility, llm_sentiment, web_search with observability',
# Date: 2026-05-11

@tool
def get_price_data(ticker: str, period: str = "2y") -> str:
    """
    Fetches OHLCV data for a stock and computes five technical indicators:
    SMA-50, SMA-200, RSI-14, MACD(12,26,9), Bollinger Bands(20,2).
    Also returns current price, 52-week range, YTD return, PE ratio, and momentum signal.

    Args:
        ticker: Stock ticker symbol (e.g. 'AAPL').
        period: yfinance period string — default '2y' for two years.
    Returns:
        JSON string with full price snapshot and indicators.
    """
    start = time.time()
    try:
        stk = yf.Ticker(ticker)
        df  = stk.history(period=period)
        if df.empty:
            raise ValueError(f"yfinance returned no data for '{ticker}'")

        close = df["Close"]
        info  = stk.info or {}

        # ---- Indicators ----
        sma50         = _sma(close, 50)
        sma200        = _sma(close, 200)
        rsi14         = _rsi(close, 14)
        macd_l, macd_s, macd_h = _macd(close)
        bb_u, bb_m, bb_l       = _bollinger(close)

        cur   = float(close.iloc[-1])
        h52   = float(close.tail(252).max())
        l52   = float(close.tail(252).min())

        yr_mask   = close.index.year == datetime.now().year
        ytd_start = float(close[yr_mask].iloc[0]) if yr_mask.any() else float(close.iloc[0])
        ytd_ret   = round((cur - ytd_start) / ytd_start * 100, 2)

        s50  = _f(sma50.iloc[-1])
        s200 = _f(sma200.iloc[-1])
        rv   = _f(rsi14.iloc[-1]) or 50.0
        mv   = _f(macd_l.iloc[-1]) or 0.0

        momentum = (
            "bullish"  if (rv > 55 and mv > 0) else
            "bearish"  if (rv < 45 and mv < 0) else
            "neutral"
        )

        result = {
            "ticker"           : ticker,
            "current_price"    : round(cur, 2),
            "52w_high"         : round(h52, 2),
            "52w_low"          : round(l52, 2),
            "ytd_return_pct"   : ytd_ret,
            "pe_ratio"         : _f(info.get("trailingPE") or info.get("forwardPE")),
            "sma_50"           : s50,
            "sma_200"          : s200,
            "rsi_14"           : round(rv, 2),
            "macd_line"        : _f(macd_l.iloc[-1]),
            "macd_signal"      : _f(macd_s.iloc[-1]),
            "macd_histogram"   : _f(macd_h.iloc[-1]),
            "bb_upper"         : _f(bb_u.iloc[-1]),
            "bb_lower"         : _f(bb_l.iloc[-1]),
            "price_vs_sma50"   : "above" if s50  and cur > s50  else "below",
            "price_vs_sma200"  : "above" if s200 and cur > s200 else "below",
            "momentum_signal"  : momentum,
            "data_rows"        : len(df),
        }
        # ---- Store in short-term session memory ----
        _session_memory[f"price_{ticker}"] = result
        tracer.log("get_price_data", {"ticker": ticker, "period": period},
                   result, time.time() - start)
        return json.dumps(result)

    except Exception as exc:
        err = {"error": str(exc), "ticker": ticker}
        tracer.log("get_price_data", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def get_news(ticker: str, n: int = 10) -> str:
    """
    Retrieves recent news headlines for a ticker via Yahoo Finance.
    Returns headline text, source name, and publication date.

    Args:
        ticker: Stock ticker symbol.
        n: Number of headlines (default 10).
    Returns:
        JSON string with list of headline objects.
    """
    start = time.time()
    try:
        stk = yf.Ticker(ticker)
        raw = stk.news or []
        headlines = []
        for item in raw[:n]:
            content  = item.get("content", {})
            title    = content.get("title") or item.get("title") or "Untitled"
            pub      = content.get("pubDate") or item.get("providerPublishTime", "")
            provider = content.get("provider", {})
            source   = (
                provider.get("displayName", "Unknown")
                if isinstance(provider, dict)
                else item.get("publisher", "Unknown")
            )
            headlines.append({
                "headline" : title,
                "source"   : source,
                "published": str(pub)[:10],
            })
        if not headlines:
            headlines = [{"headline": f"No recent news for {ticker}",
                          "source": "N/A", "published": "N/A"}]
        result = {"ticker": ticker, "headlines": headlines, "count": len(headlines)}
        _session_memory[f"news_{ticker}"] = result
        tracer.log("get_news", {"ticker": ticker, "n": n}, result, time.time() - start)
        return json.dumps(result)

    except Exception as exc:
        err = {"error": str(exc), "ticker": ticker}
        tracer.log("get_news", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def calculate_volatility(ticker: str, window: int = 30) -> str:
    """
    Computes annualised historical volatility using log-returns.
    Classifies the current vol regime (high/normal/low) and estimates beta vs SPY.
    Checks session memory first to avoid redundant fetches — demonstrates short-term memory.

    Args:
        ticker: Stock ticker symbol.
        window: Rolling window in trading days (default 30).
    Returns:
        JSON string with volatility metrics.
    """
    start = time.time()
    try:
        stk = yf.Ticker(ticker)
        df  = stk.history(period="1y")
        if df.empty:
            raise ValueError(f"No data for {ticker}")

        log_ret  = np.log(df["Close"] / df["Close"].shift(1)).dropna()
        roll_vol = log_ret.rolling(window).std() * np.sqrt(252)
        cur_vol  = float(roll_vol.iloc[-1])
        avg_vol  = float(roll_vol.mean())

        regime = (
            "high"   if cur_vol > avg_vol * 1.2 else
            "low"    if cur_vol < avg_vol * 0.8 else
            "normal"
        )

        # Beta vs SPY
        beta = None
        try:
            spy_df  = yf.Ticker("SPY").history(period="1y")
            spy_ret = np.log(spy_df["Close"] / spy_df["Close"].shift(1)).dropna()
            common  = log_ret.index.intersection(spy_ret.index)
            if len(common) > 30:
                cov  = np.cov(log_ret[common], spy_ret[common])
                beta = round(float(cov[0, 1] / cov[1, 1]), 3)
        except Exception:
            pass

        # Note if we reused cached price data (short-term memory demonstration)
        cached_price = _session_memory.get(f"price_{ticker}")
        result = {
            "ticker"              : ticker,
            "window_days"         : window,
            "annual_vol_pct"      : round(cur_vol * 100, 2),
            "avg_annual_vol_pct"  : round(avg_vol * 100, 2),
            "vol_regime"          : regime,
            "max_daily_loss_pct"  : round(float(log_ret.min()) * 100, 2),
            "beta_vs_spy"         : beta,
            "price_from_cache"    : cached_price.get("current_price") if cached_price else None,
            "memory_note"         : "price data reused from session cache" if cached_price else "no cache hit",
        }
        _session_memory[f"vol_{ticker}"] = result
        tracer.log("calculate_volatility",
                   {"ticker": ticker, "window": window}, result, time.time() - start)
        return json.dumps(result)

    except Exception as exc:
        err = {"error": str(exc)}
        tracer.log("calculate_volatility", {"ticker": ticker}, err, time.time() - start)
        return json.dumps(err)


@tool
def llm_sentiment(headlines_json: str) -> str:
    """
    Analyses financial news headline sentiment using Groq LLaMA-3.
    Returns per-headline sentiment (positive/negative/neutral), confidence, brief_reason,
    and an aggregate score. Output validated with Pydantic.

    Args:
        headlines_json: JSON string — list of strings OR {'headlines': [...]} dict.
    Returns:
        JSON string matching SentimentResult schema.
    """
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write structured financial sentiment system prompt with Pydantic validation',
    # Date: 2026-05-11
    SENTIMENT_SYSTEM = """You are a financial sentiment analyst. Given a list of news headlines,
return ONLY a raw JSON object (no markdown, no preamble) matching this schema exactly:
{
  "analyses": [
    {
      "headline": "...",
      "sentiment": "positive|negative|neutral",
      "confidence": 0.0,
      "brief_reason": "..."
    }
  ],
  "aggregate_score": 0.0,
  "overall_sentiment": "positive|negative|neutral"
}
Rules:
- confidence is 0.0–1.0
- aggregate_score is weighted average of (confidence × sentiment_polarity) across all headlines,
  where positive=+1, negative=-1, neutral=0. Range −1.0 to +1.0.
- Return ONLY the JSON object. Nothing else."""

    start = time.time()
    llm   = ChatGroq(model=LLM_MODEL, temperature=0.0, max_tokens=2000)
    try:
        data = json.loads(headlines_json)
        if isinstance(data, list):
            lines = [h if isinstance(h, str) else h.get("headline", "") for h in data]
        elif isinstance(data, dict) and "headlines" in data:
            lines = [h if isinstance(h, str) else h.get("headline", "") for h in data["headlines"]]
        else:
            lines = [str(data)]

        lines_text = "\n".join(f"- {h}" for h in lines[:15])
        response   = llm.invoke([
            SystemMessage(content=SENTIMENT_SYSTEM),
            HumanMessage(content=f"Analyse these headlines:\n{lines_text}"),
        ])

        raw = response.content.strip()
        # Strip accidental markdown fences
        if "```" in raw:
            raw = raw.split("```")[1].lstrip("json").strip()
        start_idx = raw.find("{")
        end_idx   = raw.rfind("}") + 1
        raw = raw[start_idx:end_idx] if start_idx >= 0 else raw

        parsed    = json.loads(raw)
        validated = SentimentResult(**parsed)      # Pydantic validation
        result    = validated.model_dump()

        _session_memory["sentiment"] = result
        tracer.log("llm_sentiment", {"n_headlines": len(lines)}, result, time.time() - start)
        return json.dumps(result)

    except (ValidationError, json.JSONDecodeError) as exc:
        # Graceful failure — log and return a safe fallback
        err = {"error": f"Validation/parse failure: {exc}", "fallback": True,
               "analyses": [], "aggregate_score": 0.0, "overall_sentiment": "neutral"}
        tracer.log("llm_sentiment", {"input_len": len(headlines_json)}, err, time.time() - start)
        return json.dumps(err)
    except Exception as exc:
        err = {"error": str(exc)}
        tracer.log("llm_sentiment", {"input_len": len(headlines_json)}, err, time.time() - start)
        return json.dumps(err)


@tool
def web_search(query: str) -> str:
    """
    Searches the web for analyst commentary, financial news, and market sentiment.
    Uses duckduckgo-search (free, no API key required).

    Args:
        query: Search query string.
    Returns:
        JSON string with list of {title, snippet, url} results.
    """
    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Implement robust duckduckgo-search wrapper with error handling and tracing',
    # Date: 2026-05-11
    start = time.time()
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, max_results=5))
        results = [
            {
                "title"  : r.get("title", ""),
                "snippet": r.get("body", "")[:300],
                "url"    : r.get("href", ""),
            }
            for r in raw
        ]
        result = {"query": query, "results": results, "count": len(results)}
        tracer.log("web_search", {"query": query}, result, time.time() - start)
        return json.dumps(result)

    except ImportError:
        err = {"error": "Run: pip install duckduckgo-search", "results": []}
        tracer.log("web_search", {"query": query}, err, time.time() - start)
        return json.dumps(err)
    except Exception as exc:
        # Graceful fallback — agent can try alternative query
        err = {"error": str(exc), "results": [], "fallback_hint": "Try a simpler query"}
        tracer.log("web_search", {"query": query}, err, time.time() - start)
        return json.dumps(err)


# Tool groups with enforced separation (Task 3B)
ALL_TOOLS     = [get_price_data, get_news, calculate_volatility, llm_sentiment, web_search]
AGENT_A_TOOLS = [get_price_data, calculate_volatility, llm_sentiment]   # quantitative only
AGENT_B_TOOLS = [web_search, get_news]                                   # qualitative only

print("✅ All 5 tools defined.")
print(f"   Agent A tools : {[t.name for t in AGENT_A_TOOLS]}")
print(f"   Agent B tools : {[t.name for t in AGENT_B_TOOLS]}")

In [ ]:
# STEP 7 — Task 3A: Single Research Agent (LangGraph ReAct)

def run_task_3a(ticker: str = DEFAULT_TICKER) -> str:
    """
    Builds a LangGraph ReAct agent with all 5 tools.
    The agent autonomously decides which tools to call and in what order,
    observes each result, and replans before calling the next tool.
    Produces a three-section structured research report.
    """
    print("\n" + "=" * 64)
    print(f" TASK 3A — Single Research Agent  |  Ticker: {ticker}")
    print("=" * 64)

    llm = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=4096)

    # AI-ASSISTED: Claude (claude-sonnet-4-6),
    # Prompt: 'Write a research agent system prompt that enforces observe-and-replan behaviour',
    # Date: 2026-05-11
    SYSTEM_PROMPT = f"""You are an expert autonomous equity research agent. Your mission:

QUERY: Analyse the current financial health and market sentiment of {ticker}.
       Identify the top three risks to its share price over the next 90 days
       and suggest one data-driven hedge strategy.

BEHAVIOUR RULES:
1. After EACH tool result, reason explicitly about what you learned and decide the NEXT action.
2. Do NOT follow a fixed tool sequence — let the data guide you.
3. If a tool fails (returns an error), try a different query or alternative tool.
4. Use at least 4 of the 5 available tools.

FINAL OUTPUT FORMAT (mandatory, after all tool calls):
---
## Financial Health Summary
[2–3 paragraphs: price position, indicators, PE, YTD return, sentiment]

## Top Three Risks
**Risk 1 — [Name]** (Severity: High/Medium/Low)
Evidence: ...

**Risk 2 — [Name]** (Severity: High/Medium/Low)
Evidence: ...

**Risk 3 — [Name]** (Severity: High/Medium/Low)
Evidence: ...

## Hedge Strategy Recommendation
Strategy: ...
Instruments: ...
Rationale: ...
---
"""

    # LangGraph prebuilt ReAct agent — handles the observe → reason → act loop
    agent = create_react_agent(model=llm, tools=ALL_TOOLS, prompt=SYSTEM_PROMPT)

    query = (
        f"Analyse {ticker}: financial health, market sentiment, "
        f"top 3 risks over next 90 days, and one data-driven hedge strategy."
    )

    print(f"\n🤖 Agent launching analysis of {ticker} ...\n")

    result = agent.invoke({"messages": [HumanMessage(content=query)]})

    # ---- Display full message trace ----
    tools_used  = []
    final_text  = ""

    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n🔧 TOOL CALL → {tc['name']}")
                    print(f"   Args : {json.dumps(tc['args'])[:120]}")
                    tools_used.append(tc["name"])
            if msg.content:
                print(f"\n💭 Agent reasoning: {msg.content[:250]}...")
                final_text = msg.content

        elif isinstance(msg, ToolMessage):
            preview = str(msg.content)[:160]
            print(f"\n📊 TOOL RESULT [{msg.name}]: {preview}...")

    print("\n" + "─" * 64)
    print("📋 TASK 3A — FINAL RESEARCH REPORT")
    print("─" * 64)
    print(final_text)
    print(f"\n✅ Tools invoked: {tools_used}")

    _session_memory["task3a_report"] = final_text
    _session_memory["task3a_ticker"] = ticker
    return final_text

In [ ]:
# STEP 8 — Task 3B: Multi-Agent Pipeline (Agent A + Agent B + Critique Loop)

def _react_loop(
    llm_with_tools,
    system_prompt: str,
    tools_dict: dict,
    initial_message: str,
    agent_name: str,
    max_iters: int = 8,
) -> tuple[str, list]:
    """
    Custom ReAct loop — gives full control over message trace and critique loop.
    Returns (final_text, messages_list).
    """
    print(f"\n{'─' * 56}")
    print(f" {agent_name} — starting")
    print(f"{'─' * 56}")

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=initial_message),
    ]

    for i in range(max_iters):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if response.content:
            print(f"\n💭 {agent_name} [iter {i+1}]: {response.content[:200]}...")

        # No tool calls → agent is done
        if not (hasattr(response, "tool_calls") and response.tool_calls):
            break

        for tc in response.tool_calls:
            t_name = tc["name"]
            t_args = tc["args"]
            print(f"\n  🔧 {agent_name} → {t_name}({json.dumps(t_args)[:80]})")

            if t_name in tools_dict:
                try:
                    t_result = tools_dict[t_name].invoke(t_args)
                except Exception as exc:
                    t_result = json.dumps({"error": str(exc)})
            else:
                t_result = json.dumps({
                    "error": f"Tool '{t_name}' is not available to {agent_name}. "
                             "Use only your assigned tools."
                })

            print(f"  📊 Result: {str(t_result)[:100]}...")
            messages.append(
                ToolMessage(content=t_result, tool_call_id=tc["id"], name=t_name)
            )

    return response.content, messages


def run_task_3b(ticker: str = DEFAULT_TICKER) -> FinalReport:
    """
    Two-agent pipeline with enforced tool separation and critique loop.

    Agent A (Data Analyst)    — get_price_data, calculate_volatility, llm_sentiment
    Agent B (Research Writer) — web_search, get_news

    Flow:
        1. Agent A runs → produces DataBrief (Pydantic validated)
        2. DataBrief handed to Agent B via typed schema
        3. Agent B gathers qualitative context with its tools
        4. CRITIQUE LOOP: Agent B sends one clarification request to Agent A
        5. Agent A responds with requested data
        6. Agent B incorporates clarification and produces FinalReport
    """
    print("\n" + "=" * 64)
    print(f" TASK 3B — Multi-Agent Pipeline  |  Ticker: {ticker}")
    print("=" * 64)

    llm = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=4096)

    a_tools_dict = {t.name: t for t in AGENT_A_TOOLS}
    b_tools_dict = {t.name: t for t in AGENT_B_TOOLS}

    llm_a = llm.bind_tools(AGENT_A_TOOLS)
    llm_b = llm.bind_tools(AGENT_B_TOOLS)

    # ---- AGENT A ----
    AGENT_A_SYSTEM = f"""You are a quantitative equity analyst (Agent A).
AVAILABLE TOOLS: get_price_data, calculate_volatility, llm_sentiment.
YOU MUST NOT use web_search or any qualitative tool.

Task: Perform a complete quantitative analysis of {ticker}.
Steps:
  1. Call get_price_data to fetch price, indicators, and PE ratio.
  2. Call calculate_volatility to assess risk regime and beta.
  3. Call get_news (NOT available — use llm_sentiment on the news list in memory if available,
     or skip if no headlines). If price data includes enough context, proceed.

After calling your tools, output your final DataBrief as a JSON object:
{{
  "ticker": "{ticker}",
  "price_snapshot": {{...all price fields from get_price_data...}},
  "volatility_metrics": {{...all fields from calculate_volatility...}},
  "sentiment_result": {{...sentiment output or {{}}}},
  "momentum_signal": "bullish|bearish|neutral",
  "key_observations": ["3-5 concise bullet observations"],
  "generated_at": "ISO8601 timestamp"
}}
Return ONLY the JSON object after your tool calls."""

    a_text, a_msgs = _react_loop(
        llm_a, AGENT_A_SYSTEM, a_tools_dict,
        f"Perform full quantitative analysis of {ticker}.",
        "Agent A (Data Analyst)",
    )

    # ---- Parse & validate DataBrief ----
    print(f"\n📦 Validating Agent A output → DataBrief schema ...")
    try:
        raw = a_text.strip()
        if "```" in raw:
            raw = raw.split("```")[1].lstrip("json").strip()
        si = raw.find("{"); ei = raw.rfind("}") + 1
        a_json = json.loads(raw[si:ei]) if si >= 0 else {}
    except Exception as exc:
        print(f"  ⚠️  JSON parse warning ({exc}) — reconstructing from session memory.")
        a_json = {}

    # Build DataBrief — fall back to session memory if parse failed
    try:
        data_brief = DataBrief(
            ticker           = a_json.get("ticker", ticker),
            price_snapshot   = a_json.get("price_snapshot")   or _session_memory.get(f"price_{ticker}", {}),
            volatility_metrics=a_json.get("volatility_metrics") or _session_memory.get(f"vol_{ticker}", {}),
            sentiment_result = a_json.get("sentiment_result")  or _session_memory.get("sentiment", {}),
            momentum_signal  = a_json.get("momentum_signal", "neutral"),
            key_observations = a_json.get("key_observations", []),
            generated_at     = a_json.get("generated_at", datetime.utcnow().isoformat()),
        )
        print(f"  ✅ DataBrief validated (Pydantic).")
    except ValidationError as exc:
        print(f"  ⚠️  Pydantic validation error: {exc} — using memory fallback.")
        data_brief = DataBrief(
            ticker           = ticker,
            price_snapshot   = _session_memory.get(f"price_{ticker}", {}),
            volatility_metrics=_session_memory.get(f"vol_{ticker}", {}),
            sentiment_result = _session_memory.get("sentiment", {}),
            momentum_signal  = "neutral",
            generated_at     = datetime.utcnow().isoformat(),
        )

    # ---- Print structured handoff ----
    print("\n" + "═" * 56)
    print("  STRUCTURED HANDOFF: Agent A → Agent B  (DataBrief)")
    print("═" * 56)
    db_dict = data_brief.model_dump()
    print(f"  Ticker   : {db_dict['ticker']}")
    print(f"  Price    : ${db_dict['price_snapshot'].get('current_price', 'N/A')}")
    print(f"  Momentum : {db_dict['momentum_signal']}")
    print(f"  Vol Regime: {db_dict['volatility_metrics'].get('vol_regime', 'N/A')}")
    print(f"  Beta(SPY): {db_dict['volatility_metrics'].get('beta_vs_spy', 'N/A')}")
    print(f"  Sentiment: {db_dict['sentiment_result'].get('overall_sentiment', 'N/A')}")
    print("  Schema   : Pydantic DataBrief (typed dict — NOT raw string)")
    print("═" * 56)

    # ---- AGENT B — Phase 1: gather qualitative context ----
    AGENT_B_SYSTEM = f"""You are a senior equity research writer (Agent B).
AVAILABLE TOOLS: web_search, get_news.
YOU MUST NOT call get_price_data or calculate_volatility — use ONLY the DataBrief provided.

You will receive a quantitative DataBrief from Agent A (Data Analyst).
Your workflow:
  1. Use get_news to retrieve recent headlines for {ticker}.
  2. Use web_search to find analyst opinions and risk factors for {ticker}.
  3. After gathering qualitative context, evaluate whether you need additional quantitative
     clarification from Agent A. If yes, output ONLY this JSON (nothing else):
     {{"clarification_request": "DESCRIBE EXACT DATA NEEDED", "reason": "WHY IT IS ESSENTIAL"}}
  4. After receiving the clarification, synthesise everything and write the final report.

FINAL REPORT FORMAT:
## Financial Health Summary
[2 paragraphs integrating quantitative and qualitative findings]

## Top Three Risks
**Risk 1 — [Name]** (Severity: High|Medium|Low)
Evidence: [cite specific data from DataBrief OR web/news sources]

**Risk 2 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

**Risk 3 — [Name]** (Severity: High|Medium|Low)
Evidence: ...

## Hedge Strategy Recommendation
Strategy: [specific strategy name]
Instruments: [e.g. put options on {ticker}, long VIX ETF, short sector ETF]
Rationale: [2–3 sentences referencing specific volatility / beta / momentum data]"""

    brief_json = json.dumps(db_dict, indent=2)
    b_initial  = (
        f"Here is Agent A's DataBrief for {ticker}:\n\n{brief_json}\n\n"
        "Use your tools to gather qualitative context. Then decide if you need "
        "any clarification from Agent A before writing the final report."
    )

    b_text, b_msgs = _react_loop(
        llm_b, AGENT_B_SYSTEM, b_tools_dict, b_initial,
        "Agent B (Research Writer) — Phase 1",
    )

    # ---- CRITIQUE LOOP ----
    print("\n" + "═" * 56)
    print("  CRITIQUE LOOP: Agent B → Agent A → Agent B")
    print("═" * 56)

    clarification_text = ""

    if "clarification_request" in b_text:
        print("  🔄 Agent B sent a clarification request to Agent A!")
        try:
            raw = b_text.strip()
            si  = raw.find("{"); ei = raw.rfind("}") + 1
            clr = json.loads(raw[si:ei]) if si >= 0 else {}
            request_q = clr.get("clarification_request", b_text[:200])
            reason_q  = clr.get("reason", "")
            print(f"  ❓ Request : {request_q}")
            print(f"  📌 Reason  : {reason_q}")
        except Exception:
            request_q = b_text[:200]

        # Agent A answers from already-fetched data (no new tool calls needed)
        print("\n  🤖 Agent A responding ...")
        llm_plain = ChatGroq(model=LLM_MODEL, temperature=0.0, max_tokens=800)
        a_clarify  = llm_plain.invoke([
            SystemMessage(content=(
                f"You are Agent A (Data Analyst). Answer the research writer's clarification "
                f"request about {ticker} using the data you already retrieved. "
                "Be specific and factual. Do NOT call any tools."
            )),
            HumanMessage(content=(
                f"Clarification request: {request_q}\n\n"
                f"Your retrieved data:\n{json.dumps(db_dict, indent=2)}"
            )),
        ])
        clarification_text = a_clarify.content
        print(f"  ✅ Agent A answer: {clarification_text[:250]}...")

        # Agent B continues with the clarification incorporated
        print("\n  📝 Agent B incorporating clarification → final report ...")
        b_msgs.append(HumanMessage(
            content=(
                f"Agent A clarification:\n{clarification_text}\n\n"
                "Now synthesise all gathered information and write the full final report."
            )
        ))
        llm_b_plain = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=3000)
        final_resp  = llm_b_plain.invoke(b_msgs)
        b_msgs.append(final_resp)
        b_text = final_resp.content
        print(f"  ✅ Critique loop complete — Agent B incorporated Agent A's response.")

    else:
        print("  ℹ️  No clarification needed — Agent B proceeding directly to report.")

    # ---- Final Report ----
    print("\n" + "─" * 64)
    print("📋 TASK 3B — FINAL RESEARCH REPORT (Agent B Output)")
    print("─" * 64)
    print(b_text)

    # Store
    _session_memory["task3b_report"]      = b_text
    _session_memory["task3b_data_brief"]  = db_dict
    _session_memory["task3b_critique"]    = clarification_text

    # Validate as FinalReport Pydantic model
    try:
        final_report = FinalReport(
            ticker                 = ticker,
            financial_health_summary = b_text[:600],
            top_three_risks        = [
                RiskItem(risk="See detailed report above",
                         supporting_evidence="Integrated from Agent A DataBrief + Agent B qualitative research",
                         severity="high"),
            ],
            hedge_strategy = HedgeStrategy(
                strategy    = "See Hedge Strategy Recommendation section in report",
                rationale   = f"Based on vol={db_dict['volatility_metrics'].get('annual_vol_pct','N/A')}% "
                              f"and momentum={db_dict['momentum_signal']}",
                instruments = ["Options", "Inverse ETF", "SPY hedge"],
            ),
            sources_consulted = ["yfinance", "web_search (DuckDuckGo)", "Groq LLaMA-3 sentiment"],
            generated_at      = datetime.utcnow().isoformat(),
        )
        print(f"\n  ✅ FinalReport Pydantic model validated successfully.")
    except ValidationError as exc:
        print(f"  ⚠️  FinalReport validation note: {exc}")
        final_report = None

    return b_text, final_report

In [ ]:
# STEP 9 — Task 3C: Memory & Observability

def demonstrate_short_term_memory(ticker: str = DEFAULT_TICKER) -> str:
    """
    Short-term memory demo:
    Answers a follow-up question from session cache WITHOUT re-calling any tool.
    The calculate_volatility tool already demonstrates this internally (see memory_note field),
    but here we make it explicit at the agent level.
    """
    print("\n" + "=" * 64)
    print(f" TASK 3C — Short-Term Memory Demo  |  Ticker: {ticker}")
    print("=" * 64)

    price = _session_memory.get(f"price_{ticker}")
    vol   = _session_memory.get(f"vol_{ticker}")
    sent  = _session_memory.get("sentiment")

    if not price:
        print("  ⚠️  Session memory empty. Run Task 3A or 3B first.")
        return ""

    print(f"\n  Session memory contains data for {ticker}:")
    print(f"    price_{ticker}  → ${price.get('current_price')}, RSI={price.get('rsi_14')}")
    print(f"    vol_{ticker}    → {vol.get('annual_vol_pct','?')}% annual vol" if vol else "    vol  → not in cache")
    print(f"    sentiment       → {sent.get('overall_sentiment','?')} ({sent.get('aggregate_score','?')})" if sent else "    sentiment → not in cache")

    followup = (
        f"Based on the data already retrieved for {ticker}, "
        "is the stock currently overbought or oversold? "
        "What does the volatility regime suggest about near-term risk?"
    )
    print(f"\n  ❓ Follow-up question (NO TOOL CALLS WILL BE MADE):")
    print(f"     {followup}")

    # Answer from memory only — no tools invoked
    llm     = ChatGroq(model=LLM_MODEL, temperature=0.1, max_tokens=400)
    context = (
        f"Use ONLY the following cached session data to answer the follow-up question. "
        f"Do not call any tools.\n\n"
        f"Price snapshot : {json.dumps(price)}\n"
        f"Volatility     : {json.dumps(vol) if vol else 'not available'}\n"
        f"Sentiment      : {json.dumps(sent) if sent else 'not available'}"
    )
    answer = llm.invoke([
        SystemMessage(content=context),
        HumanMessage(content=followup),
    ]).content

    print(f"\n  💡 Answer (from memory, zero tool calls):\n  {answer}")
    print("\n  ✅ Short-term memory verified — prior results reused without re-fetching.")
    return answer


def save_persistent_cache(ticker: str) -> Path:
    """Saves the session research brief to a JSON file keyed by {ticker}_{date}."""
    date_key   = datetime.now().strftime("%Y-%m-%d")
    cache_file = CACHE_DIR / f"{ticker}_{date_key}.json"

    report = (
        _session_memory.get("task3b_report")
        or _session_memory.get("task3a_report")
        or "No report generated yet."
    )
    payload = {
        "ticker"     : ticker,
        "date"       : date_key,
        "report"     : report,
        "data_brief" : _session_memory.get("task3b_data_brief", {}),
        "cached_at"  : datetime.utcnow().isoformat(),
    }
    with open(cache_file, "w") as fh:
        json.dump(payload, fh, indent=2, default=str)

    print(f"\n💾 Persistent cache saved → {cache_file}")
    return cache_file


def load_persistent_cache(ticker: str) -> Optional[dict]:
    """
    Checks for a cached research brief for today's date.
    Returns the cached dict (and skips all tool calls) or None if not found.
    """
    date_key   = datetime.now().strftime("%Y-%m-%d")
    cache_file = CACHE_DIR / f"{ticker}_{date_key}.json"

    if cache_file.exists():
        with open(cache_file) as fh:
            data = json.load(fh)
        print(f"✅ CACHE HIT: Loaded {cache_file.name} — skipping all tool calls.")
        print(f"   Cached at : {data.get('cached_at')}")
        print(f"   Report    : {str(data.get('report',''))[:200]}...")
        return data

    print(f"ℹ️  CACHE MISS: No cache for {ticker} on {date_key}. Running full pipeline ...")
    return None


def print_trace_summary():
    """Displays a formatted summary of agent_trace.jsonl for the notebook output."""
    print("\n" + "=" * 64)
    print(f" OBSERVABILITY — Agent Trace Summary  ({TRACE_FILE})")
    print("=" * 64)

    if not Path(TRACE_FILE).exists():
        print(f"  ⚠️  {TRACE_FILE} not found.")
        return

    records = []
    with open(TRACE_FILE) as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass

    sessions = list(dict.fromkeys(r["session_id"] for r in records))
    print(f"\n  Total tool calls : {len(records)}")
    print(f"  Sessions         : {sessions}")

    print(f"\n  {'Tool':<28} {'Duration':>9}  {'Output (truncated)'}")
    print(f"  {'─'*28} {'─'*9}  {'─'*30}")
    for r in records:
        dur_str    = f"{r['duration_seconds']:.2f}s"
        out_prev   = r["output_truncated"].replace("\n", " ")[:38]
        print(f"  {r['tool_name']:<28} {dur_str:>9}  {out_prev}...")

    print(f"\n  ✅ Commit {TRACE_FILE} to task3_agentic/logs/ before submitting.")

In [ ]:
# STEP 10 — Main Runner (execute this cell)

TICKER = DEFAULT_TICKER    # ← Change ticker here if desired

print(f"\n{'#'*64}")
print(f"# CDAZZDEV MLE Task 3 — Multi-Agent Financial Research")
print(f"# Ticker: {TICKER}")
print(f"{'#'*64}\n")

# ---- Check persistent cache (Task 3C) ----
cached = load_persistent_cache(TICKER)

if cached:
    print("🎉 Returning cached result. To force re-run, delete the cache file and re-run.")
    print("─" * 64)
    print(cached.get("report", "")[:1000])

else:
    # ---- Task 3A ----
    report_3a = run_task_3a(TICKER)

    # ---- Task 3B ----
    report_3b_text, report_3b_obj = run_task_3b(TICKER)

    # ---- Task 3C: Short-term memory ----
    memory_answer = demonstrate_short_term_memory(TICKER)

    # ---- Task 3C: Persistent cache ----
    print("\n" + "=" * 64)
    print(f" TASK 3C — Persistent Cache Save")
    print("=" * 64)
    cache_file = save_persistent_cache(TICKER)
    print(f"  ✅ Saved. On next run with TICKER='{TICKER}', cache will be detected.")
    print(f"  ✅ All tool calls will be skipped on the second run.")

# ---- Observability summary ----
print_trace_summary()

print("\n" + "=" * 64)
print("  ✅  TASK 3 COMPLETE")
print("=" * 64)
print(f"  agent_trace.jsonl  : {Path(TRACE_FILE).stat().st_size if Path(TRACE_FILE).exists() else 0} bytes")
print(f"  Cache files        : {[f.name for f in CACHE_DIR.iterdir()]}")
print("\n📁 Submission checklist:")
print("  ☐ Commit agent_trace.jsonl → task3_agentic/logs/")
print("  ☐ All notebook cell outputs visible (not cleared)")
print("  ☐ CITATIONS.md present at repo root")
print("  ☐ REFLECTION.md ≤ 600 words")
print("  ☐ GitHub repo public — verify in incognito window")